<a href="https://colab.research.google.com/github/istanranjith175b-commits/Exploring-datasets/blob/main/imputing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [49]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# Your starting dataset
df = pd.DataFrame({
    "Age": [25, np.nan, 30, 45, 22],
    "Salary": [50000, 60000, np.nan, 80000, 45000],
    "Category": ["A", "B", "A", np.nan, "B"],
    "Target": [1, 0, 1, 0, 0] # Example target variable
})

# Separate features (X) and target (y)
X = df.drop(columns=["Target"])
y = df["Target"]

# Define preprocessing for numerical columns (Impute then Scale)
numeric_features = ["Age", "Salary"]
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Define preprocessing for categorical columns (Impute then Encode)
categorical_features = ["Category"]
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine them into a single preprocessor
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Create the final model pipeline
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

# Fit the entire pipeline safely
model_pipeline.fit(X, y)
print("Pipeline trained successfully without data leakage!")


Pipeline trained successfully without data leakage!


In [50]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.experimental import enable_iterative_imputer  # Required for IterativeImputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer

# =====================================================================
# 1. GENERATE A LARGE SYNTHETIC DATASET
# =====================================================================
print("Generating baseline dataset (1000 rows x 6 features)...")
X_raw, y_raw = make_classification(
    n_samples=1000,
    n_features=6,
    n_informative=4,
    n_redundant=2,
    random_state=42
)

# Convert to structured DataFrame
feature_names = [f"Feature_{i}" for i in range(X_raw.shape[1])]
df_X = pd.DataFrame(X_raw, columns=feature_names)
df_y = pd.Series(y_raw, name="Target")

# Introduce synthetic missing values completely at random (MCAR)
np.random.seed(42)
missing_rate = 0.20  # 20% of data cells will be blanked out

for col in df_X.columns:
    mask = np.random.rand(len(df_X)) < missing_rate
    df_X.loc[mask, col] = np.nan

print("\n--- Initial Dataset Missingness Count ---")
print(df_X.isna().sum())
print("-------------------------------------------\n")

# =====================================================================
# 2. DEFINE CANDIDATE IMPUTATION STRATEGIES
# =====================================================================
strategies = {
    "Mean Imputation": SimpleImputer(strategy="mean"),
    "Median Imputation": SimpleImputer(strategy="median"),
    "KNN Imputation (k=5)": KNNImputer(n_neighbors=5),
    "MICE Imputation (Linear)": IterativeImputer(max_iter=10, random_state=42)
}

results_log = []

# =====================================================================
# 3. RUN BENCHMARK CROSS-VALIDATION LOOP
# =====================================================================
print("Executing robust evaluation pipeline...")

for name, imputer in strategies.items():
    # Construct a ColumnTransformer wrapping the specific imputer
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric_impute", imputer, feature_names)
        ]
    )

    # Establish the processing sequence: Impute -> Scale -> Classify
    # Sticking inside a Pipeline completely prevents data leakage during CV loops!
    modeling_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("scaler", StandardScaler()),
            ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
        ]
    )

    # Evaluate across 5 separate stratified data folds
    cv_metrics = cross_validate(
        modeling_pipeline,
        df_X,
        df_y,
        cv=5,
        scoring=["accuracy", "f1", "roc_auc"],
        return_train_score=False
    )

    # Process summary metrics
    results_log.append({
        "Strategy": name,
        "Mean Accuracy": np.mean(cv_metrics["test_accuracy"]),
        "Accuracy Std": np.std(cv_metrics["test_accuracy"]),
        "Mean F1-Score": np.mean(cv_metrics["test_f1"]),
        "Mean ROC-AUC": np.mean(cv_metrics["test_roc_auc"])
    })

    print(f"✔️ Finished processing: {name}")

# =====================================================================
# 4. AGGREGATE AND PRINT FINAL COMPREHENSIVE OUTPUT
# =====================================================================
df_performance = pd.DataFrame(results_log).sort_values(by="Mean ROC-AUC", ascending=False)

print("\n=====================================================================")
print("                   FINAL PERFORMANCE BENCHMARK REPORT                ")
print("=====================================================================")
print(df_performance.to_string(index=False, formatters={
    "Mean Accuracy": "{:.4f}".format,
    "Accuracy Std": "{:.4f}".format,
    "Mean F1-Score": "{:.4f}".format,
    "Mean ROC-AUC": "{:.4f}".format
}))
print("=====================================================================\n")


Generating baseline dataset (1000 rows x 6 features)...

--- Initial Dataset Missingness Count ---
Feature_0    225
Feature_1    203
Feature_2    193
Feature_3    200
Feature_4    208
Feature_5    201
dtype: int64
-------------------------------------------

Executing robust evaluation pipeline...
✔️ Finished processing: Mean Imputation
✔️ Finished processing: Median Imputation
✔️ Finished processing: KNN Imputation (k=5)
✔️ Finished processing: MICE Imputation (Linear)

                   FINAL PERFORMANCE BENCHMARK REPORT                
                Strategy Mean Accuracy Accuracy Std Mean F1-Score Mean ROC-AUC
MICE Imputation (Linear)        0.8910       0.0128        0.8889       0.9504
         Mean Imputation        0.8690       0.0263        0.8644       0.9381
       Median Imputation        0.8630       0.0218        0.8582       0.9366
    KNN Imputation (k=5)        0.8700       0.0192        0.8657       0.9334

